In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss
from sklearn.preprocessing import KBinsDiscretizer, LabelEncoder
import networkx as nx
import warnings
import time
from scipy.special import rel_entr
from tqdm import tqdm

# Bayesian Network libraries
import bnlearn as bn
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

warnings.filterwarnings('ignore')
np.random.seed(42)

# ============================================================================
# 1. DATA LOADING AND PREPROCESSING
# ============================================================================

def load_and_preprocess_data(filepath):
    """Load and preprocess the fraud detection dataset"""
    print("Loading dataset...")
    df = pd.read_csv(filepath)
    
    print(f"Dataset shape: {df.shape}")
    print(f"\nColumn names:\n{df.columns.tolist()}")
    print(f"\nFirst few rows:\n{df.head()}")
    print(f"\nData types:\n{df.dtypes}")
    print(f"\nMissing values:\n{df.isnull().sum()}")
    print(f"\nFraud distribution:\n{df['Fraud_Label'].value_counts()}")
    
    # Drop Transaction_ID if exists (not useful for modeling)
    if 'Transaction_ID' in df.columns:
        df = df.drop('Transaction_ID', axis=1)
    
    return df


# ============================================================================
# 2. DISCRETIZATION
# ============================================================================

def discretize_dataset(df, n_bins=3):
    """
    Discretize continuous variables into categorical bins
    """
    print("\n" + "="*70)
    print("DISCRETIZING DATASET")
    print("="*70)
    
    df_discretized = df.copy()
    
    # Identify continuous and categorical columns
    continuous_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    
    # Remove target from continuous if present
    if 'Fraud_Label' in continuous_cols:
        continuous_cols.remove('Fraud_Label')
    
    print(f"\nContinuous columns ({len(continuous_cols)}): {continuous_cols}")
    print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
    
    # Discretize continuous variables
    discretizer = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile')
    
    print(f"\nDiscretizing {len(continuous_cols)} continuous columns...")
    for col in tqdm(continuous_cols, desc="Discretizing", ncols=100, leave=True):
        try:
            df_discretized[col] = discretizer.fit_transform(df[[col]]).astype(int)
        except Exception as e:
            print(f"\nError discretizing {col}: {e}")
    
    # Encode categorical variables
    label_encoders = {}
    print(f"\nEncoding {len(categorical_cols)} categorical columns...")
    for col in tqdm(categorical_cols, desc="Encoding", ncols=100, leave=True):
        le = LabelEncoder()
        df_discretized[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le
    
    # Ensure all values are integers for bnlearn
    df_discretized = df_discretized.astype(int)
    
    print(f"\nDiscretized dataset shape: {df_discretized.shape}")
    print(f"All columns are now numeric: {df_discretized.dtypes.unique()}")
    
    return df_discretized, label_encoders


# ============================================================================
# 3. STRUCTURE LEARNING USING BNLEARN (Hill Climb + BIC)
# ============================================================================

def learn_structure_bnlearn(data, target='Fraud_Label', methodtype='hc', scoretype='bic'):
    """
    Learn Bayesian Network structure using bnlearn library
    methodtype: 'hc' (hill climb), 'ex' (exhaustive), 'cs' (constraint-based)
    scoretype: 'bic', 'k2', 'bdeu'
    """
    print("\n" + "="*70)
    print(f"STRUCTURE LEARNING: BNLEARN ({methodtype.upper()} + {scoretype.upper()})")
    print("="*70)
    
    start_time = time.time()
    
    print(f"Learning structure with method={methodtype}, scoring={scoretype}...")
    
    # Learn structure using bnlearn
    model = bn.structure_learning.fit(
        data, 
        methodtype=methodtype,
        scoretype=scoretype,
        verbose=0
    )
    
    elapsed_time = time.time() - start_time
    
    # Extract edges
    edges = model['model_edges']
    print(f"\nStructure learning completed in {elapsed_time:.2f} seconds")
    print(f"Number of edges learned: {len(edges)}")
    print("\nLearned edges:")
    for edge in edges:
        print(f"  {edge[0]} -> {edge[1]}")
    
    return model, edges, elapsed_time


# ============================================================================
# 4. PARAMETER LEARNING USING PGMPY (Maximum Likelihood Estimation)
# ============================================================================

def learn_parameters_pgmpy(data, edges, target='Fraud_Label'):
    """
    Learn parameters (CPTs) using pgmpy's Maximum Likelihood Estimator
    """
    print("\n" + "="*70)
    print("PARAMETER LEARNING: PGMPY (Maximum Likelihood Estimation)")
    print("="*70)
    
    start_time = time.time()
    
    # Create BayesianNetwork model from edges
    model = BayesianNetwork(edges)
    
    print("Fitting model with Maximum Likelihood Estimation...")
    
    # Fit using Maximum Likelihood Estimator
    model.fit(data, estimator=MaximumLikelihoodEstimator)
    
    elapsed_time = time.time() - start_time
    
    print(f"\nParameter learning completed in {elapsed_time:.2f} seconds")
    print(f"Number of CPTs learned: {len(model.get_cpds())}")
    
    # Print target CPT if available
    if target in [cpd.variable for cpd in model.get_cpds()]:
        print(f"\nCPT for target variable '{target}':")
        target_cpd = model.get_cpds(target)
        print(target_cpd)
    
    return model, elapsed_time


# ============================================================================
# 5. INFERENCE USING PGMPY (Variable Elimination)
# ============================================================================

def predict_proba_pgmpy(test_data, model, target='Fraud_Label'):
    """
    Predict probabilities using pgmpy's Variable Elimination inference
    """
    predictions = []
    
    # Create inference object
    inference = VariableElimination(model)
    
    # Get all variables except target
    evidence_vars = [col for col in test_data.columns if col != target]
    
    print(f"\nPerforming inference on {len(test_data)} samples...")
    
    for idx, row in tqdm(test_data.iterrows(), total=len(test_data), 
                         desc="Predicting", ncols=100, leave=True):
        try:
            # Create evidence dictionary
            evidence = {var: int(row[var]) for var in evidence_vars if var in model.nodes()}
            
            # Query the target variable
            result = inference.query(variables=[target], evidence=evidence, show_progress=False)
            
            # Get probability of fraud (class 1)
            prob_fraud = result.values[1] if len(result.values) > 1 else 0.5
            predictions.append(prob_fraud)
            
        except Exception as e:
            # If inference fails, use default probability
            predictions.append(0.5)
    
    return np.array(predictions)


def predict_pgmpy(test_data, model, target='Fraud_Label', threshold=0.5):
    """Predict class labels"""
    proba = predict_proba_pgmpy(test_data, model, target)
    return (proba >= threshold).astype(int)


# ============================================================================
# 6. EVALUATION METRICS
# ============================================================================

def calculate_kl_divergence(y_true, y_pred_proba):
    """Calculate Kullback-Leibler Divergence"""
    # Create probability distributions
    y_true_proba = np.zeros((len(y_true), 2))
    y_true_proba[np.arange(len(y_true)), y_true] = 1
    
    y_pred_dist = np.column_stack([1 - y_pred_proba, y_pred_proba])
    
    # Add small epsilon to avoid log(0)
    epsilon = 1e-10
    y_true_proba = y_true_proba + epsilon
    y_pred_dist = y_pred_dist + epsilon
    
    # Normalize
    y_true_proba = y_true_proba / y_true_proba.sum(axis=1, keepdims=True)
    y_pred_dist = y_pred_dist / y_pred_dist.sum(axis=1, keepdims=True)
    
    # Calculate KL divergence
    kl = np.sum(rel_entr(y_true_proba, y_pred_dist))
    return kl / len(y_true)


def calculate_ece(y_true, y_pred_proba, n_bins=10):
    """Calculate Expected Calibration Error"""
    bins = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.digitize(y_pred_proba, bins) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)
    
    ece = 0
    for i in range(n_bins):
        mask = bin_indices == i
        if np.sum(mask) > 0:
            bin_accuracy = np.mean(y_true[mask])
            bin_confidence = np.mean(y_pred_proba[mask])
            bin_weight = np.sum(mask) / len(y_true)
            ece += bin_weight * np.abs(bin_accuracy - bin_confidence)
    
    return ece


def evaluate_model(y_true, y_pred, y_pred_proba):
    """Calculate all evaluation metrics"""
    metrics = {}
    
    # Classification accuracy
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    
    # AUC score
    metrics['auc'] = roc_auc_score(y_true, y_pred_proba)
    
    # Brier score
    metrics['brier_score'] = brier_score_loss(y_true, y_pred_proba)
    
    # KL Divergence
    metrics['kl_divergence'] = calculate_kl_divergence(y_true, y_pred_proba)
    
    # Expected Calibration Error
    metrics['ece'] = calculate_ece(y_true, y_pred_proba)
    
    return metrics


# ============================================================================
# 7. CROSS-VALIDATION
# ============================================================================

def cross_validate_bn(data, target='Fraud_Label', k=5, methodtype='hc', scoretype='bic'):
    """
    Perform K-fold cross-validation for Bayesian Network
    """
    print("\n" + "="*70)
    print(f"CROSS-VALIDATION (K={k})")
    print("="*70)
    
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    
    cv_results = {
        'accuracy': [],
        'auc': [],
        'brier_score': [],
        'kl_divergence': [],
        'ece': [],
        'train_time': [],
        'test_time': []
    }
    
    fold = 1
    final_model = None
    final_edges = None
    
    for train_idx, test_idx in kf.split(data):
        print(f"\n{'='*70}")
        print(f"FOLD {fold}/{k}")
        print(f"{'='*70}")
        
        train_data = data.iloc[train_idx].reset_index(drop=True)
        test_data = data.iloc[test_idx].reset_index(drop=True)
        
        print(f"Train size: {len(train_data)}, Test size: {len(test_data)}")
        
        # Structure learning with bnlearn
        bn_model, edges, struct_time = learn_structure_bnlearn(
            train_data, 
            target=target, 
            methodtype=methodtype,
            scoretype=scoretype
        )
        
        # Parameter learning with pgmpy
        pgmpy_model, param_time = learn_parameters_pgmpy(train_data, edges, target)
        
        train_time = struct_time + param_time
        
        # Prediction on test set
        print(f"\nTesting on fold {fold}...")
        test_start = time.time()
        y_true = test_data[target].values
        y_pred_proba = predict_proba_pgmpy(test_data, pgmpy_model, target)
        y_pred = (y_pred_proba >= 0.5).astype(int)
        test_time = time.time() - test_start
        
        # Evaluate
        print(f"\nCalculating metrics...")
        metrics = evaluate_model(y_true, y_pred, y_pred_proba)
        
        print(f"\n{'='*70}")
        print(f"FOLD {fold} RESULTS:")
        print(f"{'='*70}")
        print(f"  Accuracy:       {metrics['accuracy']:.4f}")
        print(f"  AUC:            {metrics['auc']:.4f}")
        print(f"  Brier Score:    {metrics['brier_score']:.4f}")
        print(f"  KL Divergence:  {metrics['kl_divergence']:.4f}")
        print(f"  ECE:            {metrics['ece']:.4f}")
        print(f"  Train Time:     {train_time:.2f}s")
        print(f"  Test Time:      {test_time:.2f}s")
        
        cv_results['accuracy'].append(metrics['accuracy'])
        cv_results['auc'].append(metrics['auc'])
        cv_results['brier_score'].append(metrics['brier_score'])
        cv_results['kl_divergence'].append(metrics['kl_divergence'])
        cv_results['ece'].append(metrics['ece'])
        cv_results['train_time'].append(train_time)
        cv_results['test_time'].append(test_time)
        
        # Save last fold model for visualization
        final_model = bn_model
        final_edges = edges
        
        fold += 1
    
    return cv_results, final_model, final_edges


# ============================================================================
# 8. VISUALIZATION
# ============================================================================

def visualize_dag(edges, all_nodes, target='Fraud_Label', save_path='bn_dag.png'):
    """Visualize the learned DAG structure"""
    print("\nVisualizing DAG...")
    
    G = nx.DiGraph()
    
    # Add all nodes
    G.add_nodes_from(all_nodes)
    
    # Add edges
    G.add_edges_from(edges)
    
    # Create figure
    plt.figure(figsize=(16, 12))
    
    # Use spring layout for better visualization
    pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    
    # Color nodes: target in red, others in blue
    node_colors = ['red' if node == target else 'lightblue' for node in G.nodes()]
    
    # Draw network
    nx.draw(G, pos, 
            with_labels=True, 
            node_color=node_colors,
            node_size=3000,
            font_size=8,
            font_weight='bold',
            arrows=True,
            arrowsize=20,
            edge_color='gray',
            width=2,
            arrowstyle='->',
            connectionstyle='arc3,rad=0.1')
    
    plt.title("Learned Bayesian Network Structure (DAG)", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"DAG saved to {save_path}")
    plt.show()


def visualize_dag_bnlearn(bn_model, save_path='bn_dag_bnlearn.png'):
    """Visualize DAG using bnlearn's built-in visualization"""
    print("\nVisualizing DAG with bnlearn...")
    
    try:
        bn.plot(bn_model, verbose=0)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"DAG saved to {save_path}")
        plt.show()
    except Exception as e:
        print(f"Error plotting with bnlearn: {e}")


def plot_cv_results(cv_results, save_path='cv_results.png'):
    """Plot cross-validation results"""
    print("\nPlotting cross-validation results...")
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('Cross-Validation Results', fontsize=16, fontweight='bold')
    
    metrics = ['accuracy', 'auc', 'brier_score', 'kl_divergence', 'ece', 'train_time']
    titles = ['Accuracy', 'AUC', 'Brier Score', 'KL Divergence', 'ECE', 'Train Time (s)']
    
    for idx, (metric, title) in enumerate(zip(metrics, titles)):
        ax = axes[idx // 3, idx % 3]
        values = cv_results[metric]
        
        ax.bar(range(1, len(values) + 1), values, color='steelblue', alpha=0.7)
        ax.axhline(np.mean(values), color='red', linestyle='--', 
                   label=f'Mean: {np.mean(values):.4f}')
        ax.set_xlabel('Fold', fontweight='bold')
        ax.set_ylabel(title, fontweight='bold')
        ax.set_title(title, fontweight='bold')
        ax.legend()
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"CV results plot saved to {save_path}")
    plt.show()


def print_final_results(cv_results):
    """Print final summary statistics"""
    print("\n" + "="*70)
    print("FINAL RESULTS - SUMMARY STATISTICS")
    print("="*70)
    
    metrics = ['accuracy', 'auc', 'brier_score', 'kl_divergence', 'ece', 'train_time', 'test_time']
    titles = ['Accuracy', 'AUC', 'Brier Score', 'KL Divergence', 'ECE', 'Train Time (s)', 'Test Time (s)']
    
    print(f"\n{'Metric':<20} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
    print("-" * 70)
    
    for metric, title in zip(metrics, titles):
        values = cv_results[metric]
        mean_val = np.mean(values)
        std_val = np.std(values)
        min_val = np.min(values)
        max_val = np.max(values)
        
        print(f"{title:<20} {mean_val:<12.4f} {std_val:<12.4f} {min_val:<12.4f} {max_val:<12.4f}")


# ============================================================================
# 9. MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function"""
    print("="*70)
    print("DISCRETE BAYESIAN NETWORK - FRAUD DETECTION")
    print("Using: bnlearn (Structure Learning) + pgmpy (Parameter Learning & Inference)")
    print("="*70)
    
    # Load data
    filepath = '../dataset/synthetic_fraud_dataset.csv'  # UPDATE THIS PATH
    df = load_and_preprocess_data(filepath)
    
    # Discretize dataset
    df_discretized, label_encoders = discretize_dataset(df, n_bins=4)
    
    # Perform cross-validation
    cv_results, final_model, final_edges = cross_validate_bn(
        df_discretized, 
        target='Fraud_Label', 
        k=5,
        methodtype='hc',  # hill climb
        scoretype='bic'   # BIC scoring
    )
    
    # Print final results
    print_final_results(cv_results)
    
    # Visualizations
    print("\n" + "="*70)
    print("GENERATING VISUALIZATIONS")
    print("="*70)
    
    # Visualize with networkx
    visualize_dag(final_edges, df_discretized.columns.tolist(), 
                  target='Fraud_Label', save_path='bn_dag.png')
    
    # Visualize with bnlearn
    visualize_dag_bnlearn(final_model, save_path='bn_dag_bnlearn.png')
    
    # Plot CV results
    plot_cv_results(cv_results, save_path='cv_results.png')
    
    print("\n" + "="*70)
    print("ANALYSIS COMPLETE!")
    print("="*70)
    
    return cv_results, final_model, final_edges, df_discretized


if __name__ == "__main__":
    # Run the complete pipeline
    cv_results, bn_model, edges, discretized_data = main()